In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
 
# Ensure the .env file exists and write the key to it
with open('.env', 'a') as f:
    f.write(f'GROQ_API_KEY="{GROQ_API_KEY}"\n')
 
print("GROQ_API_KEY written to .env file.")
 
import os, json
from dotenv import load_dotenv
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq
 
 
load_dotenv()
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.1-8b-instant",
    groq_api_key=GROQ_API_KEY
)
 
# --- shared state schema ---
class AgentState(TypedDict):
    goal:       str
    tasks:      List[str]
    results:    List[str]
    critique:   str
    approved:   bool
    iterations: int
 
def planner(state: AgentState) -> AgentState:
    system = """You are a planning agent. Break the user's goal into
at most 5 concrete, actionable tasks. Respond ONLY with a
valid JSON array of strings. No preamble, no markdown."""
 
    messages = [
        SystemMessage(content=system),
        HumanMessage(content=f"Goal: {state['goal']}")
    ]
    response = llm.invoke(messages).content.strip()
 
    try:
        clean = response.replace("```json","").replace("```","").strip()
        tasks = json.loads(clean)
    except json.JSONDecodeError:
        tasks = [response]   # fallback: treat whole response as one task
 
    print(f"\n[Planner] Generated {len(tasks)} tasks:")
    for i, t in enumerate(tasks): print(f"  {i+1}. {t}")
 
    return {**state, "tasks": tasks}
 
initial_state: AgentState = {
    "goal":       "Research and summarise the top 3 trends in agriculture for 2025",
    "tasks":      [],
    "results":    [],
    "critique":   "",
    "approved":   False,
    "iterations": 0
}
planner(initial_state)

GROQ_API_KEY written to .env file.


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}